# N12 — Local LLM/Multimodal Offline Readiness

**Sessions:** 56–57  
**Objective:** Validate local assets, produce schema-constrained outputs, and build a no-download multimodal smoke test.

## Task contract
Run top-to-bottom from a fresh kernel. Do not install packages inside the notebook. Record one error or misconception and complete the independent transfer before submission.


In [ ]:
from __future__ import annotations
import hashlib, json, platform, random
from pathlib import Path
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Python:", platform.python_version())
print("Working directory:", Path.cwd())


In [ ]:
import os, re
MODEL_DIR=Path(os.environ.get('NOAI_LOCAL_MODEL_DIR','local_models/qwen'))
required_any=['config.json','tokenizer_config.json','preprocessor_config.json']
manifest={'path':str(MODEL_DIR),'exists':MODEL_DIR.exists(),'files':sorted([p.name for p in MODEL_DIR.glob('*')])[:30] if MODEL_DIR.exists() else []}
print(json.dumps(manifest,indent=2))
print('Offline model is optional for this smoke test; scored use requires teacher-provided rule-permitted assets.')


In [ ]:
def validate_structured_output(value):
    required={'label','confidence','reason'}
    if set(value)!=required: raise ValueError(f'keys must be {sorted(required)}')
    if value['label'] not in {'true','false'}: raise ValueError('invalid label')
    if not isinstance(value['confidence'],(int,float)) or not 0<=value['confidence']<=1: raise ValueError('invalid confidence')
    if not isinstance(value['reason'],str) or len(value['reason'])>160: raise ValueError('invalid reason')
    return True

mock_output={'label':'true','confidence':0.78,'reason':'The caption and measured diagram shape agree.'}
assert validate_structured_output(mock_output)
print(json.dumps(mock_output,ensure_ascii=False))


In [ ]:
# Lightweight multimodal smoke test: text and image-derived features, no model download.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
rng=np.random.default_rng(SEED); n=300
actual_shape=rng.integers(0,3,n); stated_shape=np.where(rng.random(n)>.5,actual_shape,rng.integers(0,3,n)); size=rng.integers(0,2,n)
y=(actual_shape==stated_shape).astype(int)
text_only=np.c_[stated_shape,size]; image_only=np.c_[actual_shape,size]; fusion=np.c_[stated_shape,actual_shape,size]
tr=np.arange(0,220); va=np.arange(220,n)
for name,X in [('text',text_only),('image',image_only),('fusion',fusion)]:
    m=LogisticRegression(max_iter=1000).fit(X[tr],y[tr]); print(name,f1_score(y[va],m.predict(X[va])))


In [ ]:
asset_record={
 'model_path':str(MODEL_DIR),
 'model_files':manifest['files'],
 'sha256':{},
 'network_required':False,
 'api_keys_present':False,
 'structured_schema':['label','confidence','reason'],
}
for filename in manifest['files'][:5]:
    p=MODEL_DIR/filename
    if p.is_file() and p.stat().st_size<5_000_000:
        asset_record['sha256'][filename]=hashlib.sha256(p.read_bytes()).hexdigest()
print(json.dumps(asset_record,indent=2))


## Independent transfer
With teacher-provided local Qwen assets, run one smoke-test prompt with networking disabled. Record model/processor versions, asset hashes, peak memory, latency, deterministic settings, parsed JSON, and a failure-safe fallback. Do not commit weights or credentials.


## Fresh-run checklist

- [ ] Restart kernel and run all cells in order.
- [ ] Confirm assertions pass.
- [ ] Record package versions and seed.
- [ ] Save the required artifact with a relative path.
- [ ] Add an error-log entry and AI-use note.
- [ ] Explain one teacher-selected cell without reading it.
